# 02 — Ingesta INPC + SHCP + IGAE → Parquet (Bronce)

**Proyecto:** Mexico Economic Pipeline  
**Fuentes:** INEGI (INPC · IGAE) · SHCP (Finanzas Públicas)  
**Output:** `data/raw/inpc/parquet/` · `data/raw/shcp/parquet/` · `data/raw/igae/parquet/`

Este notebook lee los tres archivos fuente, aplica transformaciones mínimas de estructura
y los persiste como Parquet en la capa Bronce. La limpieza de contenido va en `06_transformacion.ipynb`.

---
### Estructura esperada de archivos fuente
```
data/raw/
├── inpc/
│   └── conjunto_de_datos_inpc_mensual.csv   ← BIE INEGI, formato long, 7 columnas
├── shcp/
│   ├── ingreso_gasto_finan.csv              ← SHCP Estadísticas Oportunas, reciente
│   └── ingreso_gasto_finan_hist.csv         ← histórico
└── igae/
    └── IGAE_2.xlsx                          ← INEGI, hoja "Tabulado", formato wide
```

### Formato INPC (BIE INEGI)
```
COBERTURA | PERIODICIDAD | FECHA       | CONCEPTO                        | VALOR   | UNIDAD_MEDIDA | ESTATUS
Nacional  | Mensual      | 2021-02-01  | ...Precios al Consumidor (INPC)  | 110.907 | Índice        | Cifras definitivas
```
~1,989 filas · 7 subíndices · cobertura: feb 2021 – abr 2026

### Formato SHCP (Finanzas Públicas)
16 columnas · `CICLO` (año), `MES` (nombre en español), `CLAVE_DE_CONCEPTO`, `MONTO` (miles de pesos)  
XBB = Ingresos · XBC = Gasto · Sector: Gobierno Federal Presupuestario · 1990–2026

### Formato IGAE (INEGI Excel)
Hoja `Tabulado`, formato wide: actividades como filas, periodos mensuales como columnas.  
4 series: Total · Actividades primarias · secundarias · terciarias · 1993–2026 · base 2018=100

## 0 · Imports y configuración de Spark

In [ ]:
import os
from pathlib import Path

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, DoubleType
)

spark = (
    SparkSession.builder
    .appName("ingesta_inpc_shcp_igae")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} iniciado")

In [ ]:
# ── Paths base (relativos a la raíz del repo) ────────────────────────────────
BASE  = Path("..")
RAW   = BASE / "data" / "raw"

# Fuentes INPC
INPC_CSV = str(RAW / "inpc" / "conjunto_de_datos_inpc_mensual.csv")

# Fuentes SHCP (ingreso/gasto gobierno federal)
SHCP_CSV      = str(RAW / "shcp" / "ingreso_gasto_finan.csv")
SHCP_HIST_CSV = str(RAW / "shcp" / "ingreso_gasto_finan_hist.csv")

# Fuente IGAE (Excel INEGI)
IGAE_XLSX = str(RAW / "igae" / "IGAE_2.xlsx")

# Destinos Parquet (bronce)
INPC_PARQUET = str(RAW / "inpc" / "parquet")
SHCP_PARQUET = str(RAW / "shcp" / "parquet")
IGAE_PARQUET = str(RAW / "igae" / "parquet")

# Crear directorios de salida
for d in [INPC_PARQUET, SHCP_PARQUET, IGAE_PARQUET]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("Paths configurados ✓")
print(f"  INPC fuente    : {INPC_CSV}")
print(f"  SHCP reciente  : {SHCP_CSV}")
print(f"  SHCP histórico : {SHCP_HIST_CSV}")
print(f"  IGAE Excel     : {IGAE_XLSX}")
print(f"  INPC parquet   : {INPC_PARQUET}")
print(f"  SHCP parquet   : {SHCP_PARQUET}")
print(f"  IGAE parquet   : {IGAE_PARQUET}")

---
## 1 · INPC — Índice Nacional de Precios al Consumidor

El archivo `conjunto_de_datos_inpc_mensual.csv` del BIE ya viene en formato **long**.
Columnas: `COBERTURA`, `PERIODICIDAD`, `FECHA`, `CONCEPTO`, `VALOR`, `UNIDAD_MEDIDA`, `ESTATUS`.

Pasos:
1. Leer el CSV tal cual
2. Renombrar columnas a snake_case
3. Extraer el nombre corto del subíndice desde `CONCEPTO` (última parte de la jerarquía)
4. Derivar `anio`, `mes`, `trimestre` desde `FECHA`
5. Guardar Parquet particionado por `anio`

In [ ]:
# ── 1.1 Leer INPC crudo ──────────────────────────────────────────────────────
df_inpc_raw = (
    spark.read
    .option("header", "true")
    .option("encoding", "latin1")
    .option("inferSchema", "false")
    .csv(INPC_CSV)
)

print(f"Filas    : {df_inpc_raw.count():,}")
print(f"Columnas : {df_inpc_raw.columns}")
df_inpc_raw.show(5, truncate=80)

In [ ]:
# ── 1.2 Renombrar columnas a snake_case y seleccionar las útiles ─────────────
# Columnas originales: COBERTURA, PERIODICIDAD, FECHA, CONCEPTO, VALOR, UNIDAD_MEDIDA, ESTATUS
# Nos quedamos con: FECHA, CONCEPTO, VALOR  (el resto son constantes o metadatos)

df_inpc = (
    df_inpc_raw
    .withColumnRenamed("FECHA",    "fecha")
    .withColumnRenamed("CONCEPTO", "concepto")
    .withColumnRenamed("VALOR",    "valor")
    .select("fecha", "concepto", "valor")
)

# ── 1.3 Extraer nombre corto del subíndice desde CONCEPTO ────────────────────
# CONCEPTO es una jerarquía separada por comas:
#   "Índice nacional..., Resumen, Subíndices..., Precios al Consumidor (INPC)"
#   "Índice nacional..., Resumen, Subíndices..., Subyacente"
#   "Índice nacional..., Resumen, Subíndices..., Subyacente, Mercancías"
# → tomamos la ÚLTIMA parte después de la última coma como nombre corto

df_inpc = df_inpc.withColumn(
    "subindice",
    F.trim(F.element_at(F.split(F.col("concepto"), ","), -1))
)

print("Subíndices disponibles:")
df_inpc.select("subindice").distinct().orderBy("subindice").show(truncate=False)

In [ ]:
# ── 1.4 Derivar anio, mes y trimestre desde FECHA (formato YYYY-MM-DD) ────────
df_inpc = (
    df_inpc
    .withColumn("anio",      F.year(F.to_date(F.col("fecha"), "yyyy-MM-dd")).cast(IntegerType()))
    .withColumn("mes",       F.month(F.to_date(F.col("fecha"), "yyyy-MM-dd")).cast(IntegerType()))
    .withColumn("trimestre", F.ceil(F.col("mes") / 3).cast(IntegerType()))
    .withColumn("indice_valor", F.col("valor").cast(DoubleType()))
    .drop("valor", "concepto")   # concepto ya está resumida en subindice
)

print("Schema final:")
df_inpc.printSchema()

print("\nMuestra:")
df_inpc.show(8, truncate=False)

print("\nRango temporal:")
df_inpc.select(F.min("anio"), F.max("anio"), F.countDistinct("subindice").alias("n_subindices")).show()

In [ ]:
# ── 1.5 Guardar como Parquet bronce ──────────────────────────────────────────
(
    df_inpc
    .write
    .mode("overwrite")
    .partitionBy("anio")
    .parquet(INPC_PARQUET)
)

print(f"✅ INPC guardado en: {INPC_PARQUET}")

# Verificar lectura
df_check = spark.read.parquet(INPC_PARQUET)
print(f"   Filas totales  : {df_check.count():,}")
print(f"   Columnas       : {df_check.columns}")
df_check.groupBy("anio").count().orderBy("anio").show()

---
## 2 · SHCP — Ingresos y Gastos del Gobierno Federal

Fuente: [SHCP · Estadísticas Oportunas de Finanzas Públicas](https://www.finanzaspublicas.hacienda.gob.mx/)

Dos archivos CSV:
- `ingreso_gasto_finan.csv` — datos recientes
- `ingreso_gasto_finan_hist.csv` — datos históricos

Schema (16 columnas): `CICLO, MES, CLAVE_DE_CONCEPTO, NOMBRE, TEMA, SUBTEMA, SECTOR, AMBITO,
TIPO_DE_INFORMACION, BASE_DE_REGISTRO, UNIDAD_DE_MEDIDA, PERIODO_INICIO, PERIODO_FINAL, FRECUENCIA, DIFUSION, MONTO`

**Series de interés:**
- `XBB` — Ingresos del Gobierno Federal
- `XBC` — Gasto Total del Gobierno Federal
- Sector: `Gobierno Federal Presupuestario`
- Cobertura: 1990–2026 · ~878 registros limpios

In [ ]:
# ── 2.1 Leer ambos CSV y unirlos ──────────────────────────────────────────────
df_shcp_raw1 = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "UTF-8")
    .csv(SHCP_CSV)
)

df_shcp_raw2 = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "UTF-8")
    .csv(SHCP_HIST_CSV)
)

df_shcp_crudo = df_shcp_raw1.unionByName(df_shcp_raw2, allowMissingColumns=True)

print(f"Reciente  : {df_shcp_raw1.count():,} filas")
print(f"Histórico : {df_shcp_raw2.count():,} filas")
print(f"Total     : {df_shcp_crudo.count():,} filas")
print(f"Columnas  : {df_shcp_crudo.columns}")
df_shcp_crudo.show(3, truncate=60)

In [ ]:
# ── 2.2 Filtrar series XBB (Ingresos) y XBC (Gasto) del Gobierno Federal ──────
series_filtradas = df_shcp_crudo.filter(
    F.col("CLAVE_DE_CONCEPTO").isin("XBB", "XBC") &
    (F.col("SECTOR") == "Gobierno Federal Presupuestario")
)

print(f"Filas filtradas : {series_filtradas.count():,}")
series_filtradas.select("CLAVE_DE_CONCEPTO", "NOMBRE", "SECTOR").distinct().show(truncate=False)

In [ ]:
# ── 2.3 Eliminar duplicados ────────────────────────────────────────────────────
n_antes = series_filtradas.count()
series_limpias = series_filtradas.dropDuplicates()
n_despues = series_limpias.count()

print(f"Duplicados eliminados: {n_antes - n_despues:,}  ({n_antes:,} → {n_despues:,})")
# Esperado: ~1180 → 878

In [ ]:
# ── 2.4 Convertir MES (texto en español) → MES_NUM (entero) ──────────────────
mes_num = (
    F.when(F.col("MES") == "Enero",      1)
     .when(F.col("MES") == "Febrero",    2)
     .when(F.col("MES") == "Marzo",      3)
     .when(F.col("MES") == "Abril",      4)
     .when(F.col("MES") == "Mayo",       5)
     .when(F.col("MES") == "Junio",      6)
     .when(F.col("MES") == "Julio",      7)
     .when(F.col("MES") == "Agosto",     8)
     .when(F.col("MES") == "Septiembre", 9)
     .when(F.col("MES") == "Octubre",    10)
     .when(F.col("MES") == "Noviembre",  11)
     .when(F.col("MES") == "Diciembre",  12)
)

series_limpias = series_limpias.withColumn("MES_NUM", mes_num)

print("Verificación MES → MES_NUM:")
series_limpias.select("MES", "MES_NUM").distinct().orderBy("MES_NUM").show()

In [ ]:
# ── 2.5 Construir columna FECHA y trimestre ───────────────────────────────────
series_limpias = (
    series_limpias
    .withColumn("FECHA",     F.make_date(F.col("CICLO"), F.col("MES_NUM"), F.lit(1)))
    .withColumn("TRIMESTRE", F.ceil(F.col("MES_NUM") / 3).cast(IntegerType()))
)

print("Muestra con FECHA y TRIMESTRE:")
series_limpias.select("CICLO", "MES", "MES_NUM", "FECHA", "TRIMESTRE").show(5)

print("\nRango temporal:")
series_limpias.select(
    F.min("FECHA").alias("primera"),
    F.max("FECHA").alias("ultima"),
    F.countDistinct("FECHA").alias("periodos")
).show()

In [ ]:
# ── 2.6 Seleccionar y ordenar columnas finales ────────────────────────────────
# Columnas de salida (alineadas con Lab 5):
# FECHA, CICLO, MES, MES_NUM, TRIMESTRE, CLAVE_DE_CONCEPTO, NOMBRE,
# SUBTEMA, SECTOR, UNIDAD_DE_MEDIDA, MONTO

df_shcp = series_limpias.select(
    "FECHA",
    "CICLO",
    "MES",
    "MES_NUM",
    "TRIMESTRE",
    "CLAVE_DE_CONCEPTO",
    "NOMBRE",
    "SUBTEMA",
    "SECTOR",
    "UNIDAD_DE_MEDIDA",
    "MONTO"
).orderBy("FECHA", "CLAVE_DE_CONCEPTO")

print("Schema final SHCP:")
df_shcp.printSchema()

print(f"\nTotal filas: {df_shcp.count():,}")
df_shcp.show(10, truncate=50)

In [ ]:
# ── 2.7 Verificar cobertura por serie ─────────────────────────────────────────
print("Registros por serie:")
df_shcp.groupBy("CLAVE_DE_CONCEPTO", "NOMBRE") \
       .agg(
           F.count("*").alias("n"),
           F.min("FECHA").alias("desde"),
           F.max("FECHA").alias("hasta")
       ).show(truncate=50)

In [ ]:
# ── 2.8 Guardar como Parquet bronce particionado por CICLO ───────────────────
(
    df_shcp
    .write
    .mode("overwrite")
    .partitionBy("CICLO")
    .parquet(SHCP_PARQUET)
)

print(f"✅ SHCP guardado en: {SHCP_PARQUET}")

# Verificar lectura
df_check = spark.read.parquet(SHCP_PARQUET)
print(f"   Filas totales  : {df_check.count():,}")
print(f"   Columnas       : {df_check.columns}")
df_check.groupBy("CICLO").count().orderBy("CICLO").show(40)

---
## 3 · Resumen de ingesta

# ── Resumen final ─────────────────────────────────────────────────────────────
print("=" * 60)
print("RESUMEN DE INGESTA")
print("=" * 60)

fuentes = [
    ("INPC", INPC_PARQUET, "anio"),
    ("SHCP", SHCP_PARQUET, "CICLO"),
    ("IGAE", IGAE_PARQUET, "CICLO"),
]

for nombre, parquet_path, col_anio in fuentes:
    df = spark.read.parquet(parquet_path)
    rango = df.select(
        F.min(col_anio).alias("min"),
        F.max(col_anio).alias("max")
    ).collect()[0]
    size_mb = sum(
        f.stat().st_size for f in Path(parquet_path).rglob("*.parquet")
    ) / 1_048_576
    print(f"\n{nombre}")
    print(f"  Filas     : {df.count():,}")
    print(f"  Columnas  : {df.columns}")
    print(f"  Período   : {rango['min']}–{rango['max']}")
    print(f"  Tamaño    : {size_mb:.1f} MB (Parquet snappy)")
    print(f"  Ruta      : {parquet_path}")

print("\n" + "=" * 60)
print("Siguiente paso: 05_validacion.ipynb")
spark.stop()

In [ ]:
# ── 3.1 Leer Excel con pandas (sin header fijo) ───────────────────────────────
# El tabulado del INEGI tiene títulos y notas antes de los datos reales.
# header=None conserva la estructura original para inspeccionarla.

xls = pd.ExcelFile(IGAE_XLSX)
print("Hojas disponibles:", xls.sheet_names)

raw = pd.read_excel(IGAE_XLSX, sheet_name="Tabulado", header=None)
print(f"Dimensiones del tabulado: {raw.shape}")
print("\nPrimeras 8 filas (vista de estructura):")
print(raw.iloc[:8, :6].to_string())

In [ ]:
# ── 3.2 Localizar fila de años y fila de meses ───────────────────────────────
# El Excel tiene:
#   - Fila con los años (4 dígitos)
#   - Fila con los meses (1–12 o nombres)
#   - Filas de datos con el nombre de la actividad en columna 0
# Buscamos la primera fila que contenga un año de 4 dígitos (ej. 1993)

fila_anios = None
for i, row in raw.iterrows():
    vals = [v for v in row if isinstance(v, (int, float)) and 1990 <= v <= 2030]
    if len(vals) > 10:  # muchos años → esta es la fila de períodos
        fila_anios = i
        break

print(f"Fila de años encontrada en índice: {fila_anios}")
print("Primeros 12 valores:", list(raw.iloc[fila_anios, :12]))

In [ ]:
# ── 3.3 Construir fechas mensuales desde las dos filas de cabecera ────────────
# El tabulado tiene años en fila_anios y meses en fila_anios+1.
# Columna 0 = nombre de actividad; columnas 1..N = períodos mensuales.

fila_meses = fila_anios + 1
anios_raw  = raw.iloc[fila_anios,  1:].values
meses_raw  = raw.iloc[fila_meses,  1:].values

# Construir etiquetas de fecha (YYYY-MM) solo donde ambos son numéricos
fechas = []
for a, m in zip(anios_raw, meses_raw):
    try:
        fechas.append(pd.Timestamp(year=int(a), month=int(m), day=1))
    except Exception:
        fechas.append(pd.NaT)

fechas_validas = sum(1 for f in fechas if pd.notna(f))
print(f"Columnas de período construidas: {fechas_validas}")
print(f"Primera fecha: {fechas[0]}  |  Última fecha con dato: {[f for f in fechas if pd.notna(f)][-1]}")

In [ ]:
# ── 3.4 Extraer bloque de índice y seleccionar 4 actividades ─────────────────
# Las filas de datos empiezan después de fila_meses.
# Columna 0 = ACTIVIDAD; resto = valores del índice.

ACTIVIDADES = [
    "Total",
    "Actividades primarias",
    "Actividades secundarias",
    "Actividades terciarias",
]

# Renombrar columnas: 0="ACTIVIDAD", resto=fechas construidas
col_names = ["ACTIVIDAD"] + fechas
datos_raw = raw.iloc[fila_meses + 1:].copy()
datos_raw.columns = col_names

# Limpiar nombres de actividad (strip, quitar NaN)
datos_raw["ACTIVIDAD"] = datos_raw["ACTIVIDAD"].astype(str).str.strip()

# Filtrar solo las 4 actividades de interés
igae_wide = datos_raw[datos_raw["ACTIVIDAD"].isin(ACTIVIDADES)].copy()

print(f"Filas seleccionadas: {len(igae_wide)}  (esperado: 4)")
print("Actividades:", igae_wide["ACTIVIDAD"].tolist())

In [ ]:
# ── 3.5 Transformar wide → long con melt() ───────────────────────────────────
# Columnas de período = todas excepto "ACTIVIDAD"
# Solo mantenemos fechas válidas (pd.Timestamp, no NaT)
cols_fecha = [c for c in igae_wide.columns if isinstance(c, pd.Timestamp)]

igae_long = igae_wide.melt(
    id_vars="ACTIVIDAD",
    value_vars=cols_fecha,
    var_name="FECHA",
    value_name="IGAE"
)

# Eliminar filas sin valor publicado (períodos futuros o sin dato)
igae_long = igae_long.dropna(subset=["IGAE"])
igae_long["IGAE"] = pd.to_numeric(igae_long["IGAE"], errors="coerce")
igae_long = igae_long.dropna(subset=["IGAE"])

# Derivar columnas temporales
igae_long["CICLO"]    = igae_long["FECHA"].dt.year.astype("int64")
igae_long["MES_NUM"]  = igae_long["FECHA"].dt.month.astype("int64")
igae_long["TRIMESTRE"] = ((igae_long["MES_NUM"] - 1) // 3 + 1).astype("int64")

igae_long = igae_long[["FECHA", "ACTIVIDAD", "IGAE", "CICLO", "MES_NUM", "TRIMESTRE"]]
igae_long = igae_long.sort_values(["FECHA", "ACTIVIDAD"]).reset_index(drop=True)

print(f"Registros en formato long: {len(igae_long):,}  (esperado: ~1,612)")
print(f"Rango: {igae_long['FECHA'].min()} → {igae_long['FECHA'].max()}")
print(igae_long.head(8).to_string(index=False))

In [ ]:
# ── 3.6 Convertir a Spark DataFrame y verificar schema ───────────────────────
df_igae = spark.createDataFrame(igae_long)

print("Schema Spark:")
df_igae.printSchema()

print("\nActividades y conteo:")
df_igae.groupBy("ACTIVIDAD").count().orderBy("ACTIVIDAD").show(truncate=False)

print("\nRango temporal:")
df_igae.select(
    F.min("FECHA").alias("primera"),
    F.max("FECHA").alias("ultima"),
    F.countDistinct("FECHA").alias("periodos"),
    F.count("*").alias("total_filas")
).show()

In [ ]:
# ── 3.7 Guardar como Parquet bronce particionado por CICLO ───────────────────
(
    df_igae
    .write
    .mode("overwrite")
    .partitionBy("CICLO")
    .parquet(IGAE_PARQUET)
)

print(f"✅ IGAE guardado en: {IGAE_PARQUET}")

# Verificar lectura
df_check = spark.read.parquet(IGAE_PARQUET)
print(f"   Filas totales  : {df_check.count():,}")
print(f"   Columnas       : {df_check.columns}")
df_check.groupBy("CICLO").count().orderBy("CICLO").show(40)

In [ ]:
# ── Resumen final ─────────────────────────────────────────────────────────────
print("=" * 55)
print("RESUMEN DE INGESTA")
print("=" * 55)

for nombre, parquet_path in [("INPC", INPC_PARQUET), ("SHCP", SHCP_PARQUET)]:
    df = spark.read.parquet(parquet_path)
    anios = df.select(F.min("anio").alias("min"), F.max("anio").alias("max")).collect()[0]
    size_mb = sum(
        f.stat().st_size for f in Path(parquet_path).rglob("*.parquet")
    ) / 1_048_576
    print(f"\n{nombre}")
    print(f"  Filas     : {df.count():,}")
    print(f"  Columnas  : {len(df.columns)}")
    print(f"  Período   : {anios['min']}–{anios['max']}")
    print(f"  Tamaño    : {size_mb:.1f} MB (Parquet snappy)")
    print(f"  Ruta      : {parquet_path}")

print("\n" + "=" * 55)
print("Siguiente paso: 03_ingesta_igae.ipynb")

spark.stop()